In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
books = pd.read_csv('/kaggle/input/datasets/mdhamani/goodreads-books-100k/GoodReads_100k_books.csv')

books.head()

In [ ]:
books.shape

In [ ]:
books.isna().sum()

In [ ]:
# title column has 1 null value, since for recommendation title is important, remove it 

books = books.dropna(subset=['title'])

In [ ]:
books.isna().sum()

In [ ]:
books.shape

In [ ]:
books.duplicated().sum()

# no duplicate entry

In [ ]:
# filter required columns
books = books[['author', 'title', 'genre', 'desc', 'rating', 'totalratings', 'img']]

In [ ]:
books.shape

In [ ]:
books.isna().sum()

In [ ]:
# fill description and genre column null values with empty string
books['desc'] = books['desc'].fillna('')

books['genre'] = books['genre'].fillna('')

In [ ]:
books['genre'].head(3)

In [ ]:
# replace comma with empty string
books['genre'] = books['genre'].str.replace(',', " ")

In [ ]:
books['genre'].head(3)

In [ ]:
# to improve recommendation precision, combine desc and genre
# because similarity model captures both: semantic description and categorical signal

books['content'] = books["desc"] + " " + books['genre']

In [ ]:
books.head(3)

In [ ]:
# weighted popularity score
C = books['rating'].mean()
m = books['totalratings'].quantile(0.75)

books['weighted_score'] = (books['totalratings'] / (books['totalratings'] + m)) * books['rating'] + (m / (books['totalratings'] + m)) * C

In [ ]:
books.head(3)

In [ ]:
books.shape

## Embedding
- TF-IDF
- E5

In [ ]:
# tf-idf vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.8,
    max_features=50000
)

tfidf_matrix = vectorizer.fit_transform(books["content"])

In [ ]:
tfidf_matrix.shape

In [ ]:
book_indices = pd.Series(books.index, index=books["title"]).drop_duplicates()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_similar_books(book_index, top_k=20):
    book_vector = tfidf_matrix[book_index]
    
    similarities = cosine_similarity(book_vector, tfidf_matrix)
    
    similarity_scores = list(enumerate(similarities[0]))
    
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    similarity_scores = similarity_scores[1:top_k+1]  # skip itself
    
    return similarity_scores

In [ ]:
import numpy as np
import pandas as pd
from difflib import get_close_matches
from sklearn.metrics.pairwise import cosine_similarity


class ContentBasedRecommender:
    def __init__(self, books_df, tfidf_matrix, book_indices):
        self.books = books_df.reset_index(drop=True)
        self.tfidf_matrix = tfidf_matrix
        self.book_indices = book_indices

    def get_popular_books(self, top_k=10):
        popular = self.books.sort_values(
            by="weighted_score", ascending=False
        ).head(top_k)

        return popular[["title", "author", "rating", "totalratings", "img"]]

    def recommend_similar(self, title, top_k=10):
        # Normalize input
        title = title.strip()
    
        # Case-insensitive matching
        titles_lower = self.book_indices.index.str.lower()
        title_lower = title.lower()
    
        if title_lower in titles_lower:
            idx = self.book_indices[titles_lower == title_lower].values[0]
        else:
            # Fuzzy matching
            matches = get_close_matches(title, self.book_indices.index, n=1, cutoff=0.6)
            
            if not matches:
                # fallback to popularity
                return self.get_popular_books(top_k)
            
            matched_title = matches[0]
            idx = self.book_indices[matched_title]
    
        book_vector = self.tfidf_matrix[idx]
    
        similarities = cosine_similarity(book_vector, self.tfidf_matrix).flatten()
    
        similarity_scores = list(enumerate(similarities))
    
        similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
        similarity_scores = similarity_scores[1: top_k + 1]
    
        book_indices = [i[0] for i in similarity_scores]
    
        recommended = self.books.iloc[book_indices]
    
        return recommended[
            ["title", "author", "rating", "totalratings", "img"]
        ]

In [ ]:
# instantiate model
recommender = ContentBasedRecommender(
    books_df=books,
    tfidf_matrix=tfidf_matrix,
    book_indices=book_indices
)

In [ ]:
recommender.get_popular_books(3)

In [ ]:
recommender.recommend_similar("Harry Potter", 5)

In [ ]:
recommender.recommend_similar("harry potter", 5)

In [ ]:
# see the output is not same
# conclusion: tf-idf is not that much good

# embedding based

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("intfloat/multilingual-e5-base")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("intfloat/multilingual-e5-base", device=device)

In [ ]:
# content for encoding
books["embedding_input"] = "passage: " + books["content"]

In [ ]:
# encode in batches
embeddings = model.encode(
    books["embedding_input"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

In [ ]:
embeddings.shape

In [ ]:
import numpy as np


class EmbeddingRecommender:
    def __init__(self, books_df, embeddings):
        self.books = books_df.reset_index(drop=True)
        self.embeddings = embeddings
        
        self.title_to_index = {
            title.lower(): idx
            for idx, title in enumerate(self.books["title"])
        }

    def get_popular_books(self, top_k=10):
        popular = self.books.sort_values(
            by="weighted_score", ascending=False
        ).head(top_k)

        return popular[
            ["title", "author", "rating", "totalratings", "img"]
        ]

    def recommend_similar(self, title, top_k=10):
        title_clean = title.strip().lower()

        if title_clean not in self.title_to_index:
            return self.get_popular_books(top_k)

        idx = self.title_to_index[title_clean]

        query_vector = self.embeddings[idx]

        # cosine similarity (embeddings already normalized)
        similarities = np.dot(self.embeddings, query_vector)

        similar_indices = np.argsort(similarities)[::-1][1: top_k + 1]

        recommended = self.books.iloc[similar_indices]

        return recommended[
            ["title", "author", "rating", "totalratings", "img"]
        ]

In [ ]:
recommender = EmbeddingRecommender(books, embeddings)

In [ ]:
recommender.recommend_similar("Harry Potter", 5)

In [ ]:
recommender.recommend_similar("harry potter", 5)

# save model

In [ ]:
import joblib

joblib.dump(recommender, "embedding_recommender.pkl")

In [ ]:
import os
os.path.getsize("embedding_recommender.pkl")

In [ ]:
books.columns

# test model

In [ ]:
import pandas as pd
import numpy as np

# books: your DataFrame
# embeddings: your numpy array (normalized)

# Save books
books.to_pickle("books.pkl")  # saves DataFrame

# Save embeddings
np.save("embeddings.npy", embeddings)  # saves numpy array

In [ ]:
import os
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch
import gdown


class EmbeddingRecommender:
    def __init__(self, books_path="books.pkl", embeddings_path="embeddings.npy"):
        # Google Drive File IDs
        BOOKS_FILE_ID = "1xiLi-JzVZzC5Dt7VJJT3M4HHVIy39sQM"
        EMBEDDINGS_FILE_ID = "177wmDD_c5826DKBMLqSRgCMY5ii96eVt"

        books_url = f"https://drive.google.com/uc?id={BOOKS_FILE_ID}"
        embeddings_url = f"https://drive.google.com/uc?id={EMBEDDINGS_FILE_ID}"

        if not os.path.exists(books_path):
            print("Downloading books.pkl from Google Drive...")
            gdown.download(books_url, books_path, quiet=False)

        if not os.path.exists(embeddings_path):
            print("Downloading embeddings.npy from Google Drive...")
            gdown.download(embeddings_url, embeddings_path, quiet=False)

        print("Loading books...")
        self.books = pd.read_pickle(books_path).reset_index(drop=True)

        print("Loading embeddings...")
        self.embeddings = np.load(embeddings_path)

        print("Loading embedding model (E5)...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(
            "intfloat/multilingual-e5-base",
            device=device
        )

        print("Model loaded successfully.")

    def _safe_str_contains(self, series, value):
        return series.fillna("").astype(str).str.contains(str(value), case=False, na=False)

    def apply_filters(self, df, filters=None):
        if not filters:
            return df

        filtered = df.copy()

        genre = filters.get("genre")
        author = filters.get("author")
        min_rating = filters.get("min_rating")


        if genre:
            filtered = filtered[self._safe_str_contains(filtered["genre"], genre)]

        if author:
            filtered = filtered[self._safe_str_contains(filtered["author"], author)]

        if min_rating is not None:
            filtered = filtered[filtered["rating"] >= min_rating]

        return filtered

    def _format_output(self, df):
        output_cols = [
            col for col in [
                "title",
                "author",
                "genre",
                "rating",
                "totalratings",
                "weighted_score",
                "img"
            ]
            if col in df.columns
        ]
        return df[output_cols]

    def get_popular_books(self, top_k=10, filters=None):
        filtered = self.apply_filters(self.books, filters)

        if filtered.empty:
            return pd.DataFrame()

        popular = filtered.sort_values(
            by="weighted_score",
            ascending=False
        ).head(top_k)

        return self._format_output(popular)

    def recommend_semantic(self, query=None, top_k=10, filters=None):
        # Case 3: filter only, no search
        if not query or not query.strip():
            return self.get_popular_books(top_k=top_k, filters=filters)

        filtered_books = self.apply_filters(self.books, filters)

        if filtered_books.empty:
            return pd.DataFrame()

        formatted_query = f"query: {query.strip()}"

        query_embedding = self.model.encode(
            formatted_query,
            normalize_embeddings=True
        )

        candidate_indices = filtered_books.index.to_numpy()
        candidate_embeddings = self.embeddings[candidate_indices]

        similarities = np.dot(candidate_embeddings, query_embedding)

        top_local_indices = np.argsort(similarities)[::-1][:top_k]
        top_global_indices = candidate_indices[top_local_indices]

        results = self.books.iloc[top_global_indices].copy()
        results["similarity"] = similarities[top_local_indices]

        return self._format_output(results)

    def recommend(self, query=None, top_k=10, filters=None):
        return self.recommend_semantic(query=query, top_k=top_k, filters=filters)


recommender = EmbeddingRecommender()

In [ ]:
recommender.recommend(query="books about deep learning", top_k=5)

In [ ]:
filters = {
    "genre": "technology",
    "min_rating": 4.0
}
recommender.recommend(query="books about deep learning", top_k=5, filters=filters)

In [ ]:
filters = {
    "genre": "fantasy",
    "min_rating": 4.2
}
recommender.recommend(query="", top_k=5, filters=filters)

In [ ]:
filters = {
    "genre": "action",
    "min_rating": 4.2
}
recommender.recommend(query="", top_k=5, filters=filters)